# Train the seven-emotion Hindi–English pilot

Run this notebook manually in Google Colab. The pretrained XLM-R and MuRIL encoders will each receive a seven-label head and learn from the same frozen labelled data. No AWS resources are needed.

The source audit must permit reuse, and the two-reviewer supplement must be completed first. The 84 demo phrases are not training data. French–English remains experimental until bilingual review is available.

Keep this as an interactive training notebook; the Streamlit dashboard runs on your laptop.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

uploaded = files.upload()  # Upload the source-only zip generated by training.cloud_bundle.
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload one source-only project ZIP."
destination = Path("/content").resolve()
with zipfile.ZipFile(archives[0]) as archive:
    assert sum(item.file_size for item in archive.infolist()) < 64 * 1024 * 1024, "Use the source-only ZIP, not model weights."
    for item in archive.infolist():
        target = (destination / item.filename).resolve()
        assert target.is_relative_to(destination), "Unsafe archive path."
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks are not allowed."
    archive.extractall(destination)
PROJECT = destination / "HinglishEmotionProject"
assert (PROJECT / "pyproject.toml").is_file()


## Prepare the environment and verify a GPU

Select a GPU in Runtime settings before continuing. Batch size is the number of phrases per forward pass. Gradient accumulation combines several small batches before a weight update. These settings reduce memory pressure; they do not guarantee a training time or accuracy.

In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model]"], check=True)
import torch
assert torch.cuda.is_available(), "No CUDA GPU assigned. Choose a GPU runtime or wait for availability."
print(torch.cuda.get_device_name(0))


## Persistent outputs

Mount your own Google Drive so the best checkpoint and latest epoch recovery state survive a disconnected Colab runtime. Keep enough space available for model weights and optimizer states. The recovery point is the end of an epoch, not the middle of a batch.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/HinglishEmotionPilot")
STORAGE.mkdir(parents=True, exist_ok=True)


## Upload the approved pilot CSV and its matching audit JSON

The CSV needs text, emotion, language_pair and fixed train/validation/test partitions, with source and grouping information. Upload the merged pilot CSV and audit generated by training.annotations build-pilot. The audit binds the approval to the exact CSV bytes.

In [ ]:
uploaded = files.upload()
csv_names = [name for name in uploaded if name.endswith(".csv")]
json_names = [name for name in uploaded if name.endswith(".json")]
assert len(csv_names) == len(json_names) == 1, "Upload exactly one reviewed CSV and its audit JSON."
DATA = STORAGE / "pilot_hi.csv"
AUDIT = STORAGE / "pilot_hi.json"
DATA.write_bytes(uploaded[csv_names[0]])
AUDIT.write_bytes(uploaded[json_names[0]])


## Validate and freeze the split

Training updates weights; validation chooses the checkpoint; the test set is measured after selection. Related variants must stay in one partition. Both models reuse the same manifest. The preview below creates no model weights.

In [ ]:
from training.data_workflow import prepare_data
MANIFEST = STORAGE / "emotion_frozen_split.csv"
prepared = prepare_data(DATA, "emotion", "raw", 42, MANIFEST, AUDIT, "hi-en")
print(prepared["report"])
print({name: len(part) for name, part in zip(("train", "validation", "test"), prepared["parts"])})
print("Frozen split:", prepared["manifest_sha256"])


## Inspect tokenizer coverage

Raw text retains emoji, accents, case and expressive spellings. Do not manufacture training emoji from the emotion label. Inspect how each tokenizer handles the actual input; unknown tokens are a limitation to record.

In [ ]:
from transformers import AutoTokenizer
from hinglish_emotion.model_catalog import ENCODER_BY_KEY
from hinglish_emotion.text_preparation import prepare_text
for key in ("xlmr", "muril"):
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_BY_KEY[key].model_id)
    for phrase in ("Main khush nahi hoon 😊", "movie acchaaa thi 👩🏽‍💻"):
        encoded = tokenizer(prepare_text(phrase), add_special_tokens=False)
        print(key, phrase, tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


## Train the matched comparison

An epoch is one pass through training phrases. Cross-entropy penalizes incorrect labels; backpropagation and AdamW update weights. The best validation macro-F1 checkpoint is saved. Three epochs are a starting configuration, not a promised optimum.

Re-run this cell after reconnecting to resume any existing epoch states. Dataset, split and training-setting mismatches are rejected.

In [ ]:
MODEL_ROOT = STORAGE / "models"
command = [
    sys.executable, "-m", "training.compare_encoders",
    "--task", "emotion", "--models", "xlmr", "muril",
    "--language-pair", "hi-en", "--data", str(DATA),
    "--data-audit", str(AUDIT), "--output-root", str(MODEL_ROOT),
    "--split-manifest", str(MANIFEST), "--epochs", "3",
    "--batch-size", "2", "--gradient-accumulation", "8",
    "--max-length", "128", "--learning-rate", "2e-5",
    "--seed", "42", "--split-seed", "42", "--precision", "auto",
    "--resume",
]
subprocess.run(command, check=True, cwd=PROJECT)


## Read the measured results

Macro-F1 gives all seven emotions equal importance. Accuracy alone can hide weak rare-class performance. Compare with the lexical baseline and inspect support, mistakes and the corpus scope. Do not claim French–English validation from a Hindi–English run.

In [ ]:
import json
import pandas as pd
display(pd.read_csv(MODEL_ROOT / "emotion_encoder_comparison.csv"))
for key in ("xlmr", "muril"):
    metrics = json.loads((MODEL_ROOT / f"{key}_emotion" / "metrics.json").read_text())
    print(key, "validation macro-F1:", metrics["best_validation_macro_f1"])
    print("Lexical baseline:", metrics["lexical_baseline"])
    display(pd.DataFrame(metrics["test"]["per_class"]).T)


## Export for your laptop

Inference ZIPs contain weights, tokenizer, preprocessing settings, metrics and the fine-tuned model card. They exclude optimizer states and raw training/test phrases. Extract the two folders into the models folder of your VIT project, then select the checkpoint in Emotion lab.

In [ ]:
from training.cloud_bundle import export_checkpoint
for key in ("xlmr", "muril"):
    archive = export_checkpoint(MODEL_ROOT / f"{key}_emotion", STORAGE / f"{key}_emotion.zip")
    print("Saved to Google Drive:", archive)
# Download one archive at a time. If browser download fails, download it from Drive.
files.download(str(STORAGE / "xlmr_emotion.zip"))
